In [1]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
import gc

# Data Ingestion

In [2]:
# df_credit_raw = pd.read_parquet("data/base/df_credit_sample_47_pct.parquet")
df_credit_raw = pd.read_parquet("data/base/df_credit_sample_dec24_may25.parquet")

In [3]:
# Load MCC data

mcc_df = pd.read_csv("data/map/mcc_data.csv", sep=";", dtype={"Code": str})
mcc_df.rename({"Code": "MCC"}, axis=1, inplace=True)

# Drop Cols with High Missing Value Pct

In [4]:
from src.utils import get_features_by_missing_pct

In [5]:
selected_cols, summary_df = get_features_by_missing_pct(
    df_credit_raw,
    0.99,
    ["PANNumber", "Transaction Serial No", "Transaction Datetime", "Confirmed"],
)

In [10]:
list_feat = [
    "Avg_Amt_L15M",
    "CntUnique_CardNo_by_MCC_L1D",
    "CntUnique_CardNo_by_cardAcceptor_cat_L1D",
    "Max_Amt_L15M",
    "Max_Amt_L1D",
    "Max_Amt_to_MCC_L15M",
    "Max_Amt_to_MCC_L1D",
    "Max_Amt_to_cardAcceptor_cat_L15M",
    "Max_Amt_to_cardAcceptor_cat_L1D",
    "Max_Amt_to_countryCode_L15M",
    "Max_Amt_to_countryCode_L1D",
    "Sum_Amt_to_MCC_L15M",
    "Sum_Amt_to_cardAcceptor_cat_L15M",
    "Sum_Amt_to_countryCode_L15M",
    "time_diff",
    "CardProduct",
    "CardStatus",
    "Cat Card Acceptor Name",
    "CustomerSex",
    "MCC Category",
    "POSMode",
    "TotalTrxAmount10Mi",
    "TotalTrxAmount15Mi",
    "TotalTrxAmount30Mi",
    "TotalTrxAmountL1D",
    "TotalTrxAmountL5min",
    "Transaction Amount",
]

In [9]:
pd.set_option("display.max_rows", None)
summary_df[
    ~summary_df.feature.isin(
        ["PANNumber", "Transaction Serial No", "Transaction Datetime", "Confirmed"]
        + selected_cols
    )
].reset_index(drop=True)

,feature,missing_pct
0,CustomerAvgIncome,1.000000
1,Balance,1.000000
2,AgeOfOpenAcctActiveCard,1.000000
3,TrfToBDIStaff,1.000000
4,BDIStaff,1.000000
5,AgeOfRegDateTxn,1.000000
6,IsSDBPastDue,1.000000
7,FlagOutBranch,1.000000
8,isTDHoldAmount,1.000000
9,isProgramHoldAmount,1.000000


In [10]:
# pd.set_option('display.max_columns', 100)
df_credit_raw = df_credit_raw[
    ["PANNumber", "Transaction Serial No", "Transaction Datetime", "Confirmed"]
    + selected_cols
].copy()

In [11]:
df_credit_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1398828 entries, 0 to 1398827
Data columns (total 67 columns):
 #   Column                          Non-Null Count    Dtype         
---  ------                          --------------    -----         
 0   PANNumber                       1398828 non-null  object        
 1   Transaction Serial No           1398828 non-null  int64         
 2   Transaction Datetime            1398828 non-null  datetime64[ns]
 3   Confirmed                       37902 non-null    object        
 4   Product Indicator               1219974 non-null  object        
 5   Transaction Amount              1387919 non-null  float64       
 6   MCC                             1398828 non-null  object        
 7   Country Code                    1398828 non-null  object        
 8   Card Acceptor Terminal ID       1398828 non-null  object        
 9   Card Acceptor ID                1398828 non-null  object        
 10  Card Acceptor Name              1393402 no

# Feature Engineering Logics

## Value Mapping Steps

1. MCC value mapping: 
- MCC to MCC Details
- MCC to MCC Transaction Category Code
- MCC to MCC Category
2. Value mapping Card Acceptor Name to bigger group (ecommerce, online ads, etc.)

In [3]:
from src.utils import clean_categorize_merchant_name
from tqdm import tqdm

In [9]:
# 1. Value mapping MCC_code to MCC Details, Trnx Category Code, Category

df_credit_clean = df_credit_raw.merge(
    mcc_df[["MCC", "Description", "Transaction Category Code", "MCC Category"]],
    on="MCC",
    how="left",
)
df_credit_clean.rename(
    columns={
        "Description": "MCC Details",
        "Transaction Category Code": "MCC Trnx Category Code",
    },
    inplace=True,
)

In [10]:
# 2. Value mapping Card Acceptor Name/Card Acceptor Name to bigger group (ecommerce, online ads, etc.)
tqdm.pandas(desc="Clean and Categorize Acceptor Name Progress")
df_credit_clean["Cat Card Acceptor Name"] = df_credit_clean[
    "Card Acceptor Name"
].progress_apply(clean_categorize_merchant_name)

Clean and Categorize Acceptor Name Progress: 100%|█████████████████████████| 1398828/1398828 [03:18<00:00, 7045.66it/s]


In [6]:
channel_cols = [
    "PANNumber",
    "Transaction Serial No",
    "Transaction Datetime",
    "Product Indicator",
    "Transaction Amount",
    "MCC",
    "MCC Details",
    "MCC Trnx Category Code",
    "MCC Category",
    "Country Code",
    "Card Acceptor Terminal ID",
    "Card Acceptor ID",
    "Card Acceptor Name",
    "Card Acceptor City",
    "Card Acceptor Region Code",
    "Card Acceptor Country Code",
    "Cat Card Acceptor Name",
    "Currency Code",
    "Confirmed",
]

In [12]:
# df_credit_clean.to_parquet("data/base/df_credit_clean_v2.parquet")
df_credit_clean.to_parquet("data/base/df_credit_clean_dec24_may25.parquet")

In [7]:
df_credit_clean = pd.read_parquet("data/base/df_credit_clean_dec24_may25.parquet")
df_credit_clean_dask = dd.read_parquet("data/base/df_credit_clean_dec24_may25.parquet")

In [8]:
df_credit_filter = df_credit_clean[channel_cols].copy()
df_credit_filter_dask = df_credit_clean_dask[channel_cols].copy()

## Downcasting Float Features

In [9]:
def preprocess_data(df):
    print("Starting preprocessing...")
    df_processed = df.copy()

    # identify and safely downcast float columns
    float_cols = df_processed.select_dtypes(include=["float64", "float32"]).columns
    print(f"Float columns to downcast: {len(float_cols)}")
    for col in float_cols:
        try:
            print("Start downcasting float64 to float32")
            df_processed[col] = df_processed[col].astype("float32")
        except Exception as e:
            print(f"Warning: failed to downcast {col}: {e}")

    for col in df_processed.columns:
        if isinstance(col, str):
            try:
                if df_processed[col].dtype == "object":
                    # treat pure whitespaces as missing
                    df_processed[col] = df_processed[col].replace(
                        r"^\s*$", np.nan, regex=True
                    )

                    unique_vals = df_processed[col].dropna().unique()
                    if set(unique_vals).issubset({"Y", "N"}):
                        print(f"Mapping Y/N to 1/0 in column: {col}")
                        df_processed[col] = df_processed[col].map({"N": 0, "Y": 1})
                if col.startswith("Is") and df_processed[col].dtype in [
                    "object",
                    "int64",
                ]:
                    print(f"Mapping Is* column Y/N to 1/0: {col}")
                    df_processed[col] = df_processed[col].map({"N": 0, "Y": 1})
            except Exception as e:
                print(f"Warning processing column {col}: {e}")

    print(
        "Estimated memory usage (MB):", df_processed.memory_usage(deep=True).sum() / 1e6
    )
    gc.collect()
    return df_processed

In [6]:
df_credit_filter = preprocess_data(df_credit_filter)

Starting preprocessing...
Float columns to downcast: 1
Start downcasting float64 to float32
Estimated memory usage (MB): 1369.824631


## Time Difference Features

In [10]:
from src.calculation_features import (
    prepare_dask_dataframe,
    generate_rolling_features_dask,
    generate_rolling_features,
    calculate_time_differences,
)

In [11]:
from src.credit_card_config import (
    time_shift_config,
    time_windows,
    freq_config,
    monetary_config_1,
    monetary_config_2,
    monetary_config_3,
    monetary_config_4,
    monetary_config_5,
    monetary_config_6,
    unique_count_config,
)
import warnings

warnings.filterwarnings("ignore")

all_monetary_configs = (
    monetary_config_1
    + monetary_config_2
    + monetary_config_3
    + monetary_config_4
    + monetary_config_5
    + monetary_config_6
)

In [16]:
df_time_diff = calculate_time_differences(
    df=df_credit_filter,
    datetime_col="Transaction Datetime",
    groupby_col="PANNumber",
    time_window=time_windows,
    config=time_shift_config,
)

Calculating rolling averages: 100%|██████████████████████████████████████████████████████| 7/7 [00:39<00:00,  5.57s/it]


In [11]:
# df_time_diff.to_parquet("data/feature_engineering/credit/v3/df_time_diff.parquet")
df_time_diff = pd.read_parquet(
    "data/feature_engineering/credit/v3/df_time_diff.parquet"
)

## Frequency Features

In [13]:
card_no_ever_alert = list(
    df_credit_filter[df_credit_filter["Confirmed"].isin([0, 1])].PANNumber.unique()
)
card_no_never_alert = list(
    df_credit_filter[
        ~df_credit_filter.PANNumber.isin(card_no_ever_alert)
    ].PANNumber.unique()
)

In [8]:
df_credit_filter_1 = df_credit_filter[
    df_credit_filter.PANNumber.isin(card_no_ever_alert)
]
df_credit_filter_2 = df_credit_filter[
    df_credit_filter.PANNumber.isin(card_no_never_alert)
]

In [11]:
# Calculate Frequency using Pandas
df_freq_1 = generate_rolling_features(
    df_credit_filter_1,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=freq_config,
)

Feature Config Progress: 100%|███████████████████████████████████████████████████████████| 8/8 [05:34<00:00, 41.77s/it]


In [13]:
df_freq_1.to_parquet("data/feature_engineering/credit/v3/df_freq_1.parquet")

In [9]:
# Calculate Frequency using Pandas
df_freq_2 = generate_rolling_features(
    df_credit_filter_2,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=freq_config,
)

Feature Config Progress: 100%|███████████████████████████████████████████████████████████| 8/8 [05:00<00:00, 37.58s/it]


In [10]:
df_freq_2.to_parquet("data/feature_engineering/credit/v3/df_freq_2.parquet")

In [ ]:
# frequency dataset
df_freq_1 = pd.read_parquet("data/feature_engineering/credit/v3/df_freq_1.parquet")
df_freq_2 = pd.read_parquet("data/feature_engineering/credit/v3/df_freq_2.parquet")
df_freq = pd.concat([df_freq_1, df_freq_2], axis=0)
df_freq.reset_index(drop=True, inplace=True)
df_freq.to_parquet("data/feature_engineering/credit/v3/df_freq.parquet")

del df_freq_1
del df_freq_2
del df_freq
gc.collect()

In [7]:
# Dask Data Preparation
df_credit_filter_dask = prepare_dask_dataframe(
    df_credit_filter_dask,
    datetime_col="Transaction Datetime",
    partition_size="500MB",
    target_partitions=50,
)

In [8]:
df_freq_dask = generate_rolling_features_dask(
    df_credit_filter_dask,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=freq_config,
)

Processing feature: frequency | Window:900S


KeyError: "['Transaction Datetime'] not in index"

## Monetary

In [14]:
import gc

# split the lists into 50:50
ever_alert_part_1, ever_alert_part_2 = np.array_split(card_no_ever_alert, 2)
never_alert_part_1, never_alert_part_2 = np.array_split(card_no_never_alert, 2)

# create dictionary to store the splits and their labels for iteration
splits = {
    "ever_alert_part_1": ever_alert_part_1,
    "ever_alert_part_2": ever_alert_part_2,
    "never_alert_part_1": never_alert_part_1,
    "never_alert_part_2": never_alert_part_2,
}

# loop over splits, filter dataframe, run feature generation, and save parquet
for (
    name,
    pan_list,
) in splits.items():
    print(f"Processing {name}...")
    df_filtered = df_credit_filter[df_credit_filter.PANNumber.isin(pan_list)]

    # run feature engineering
    df_monetary = generate_rolling_features(
        df_filtered,
        datetime_col="Transaction Datetime",
        key_col="Transaction Serial No",
        features_config=all_monetary_configs,
    )

    # save parquet
    output_path = f"data/feature_engineering/credit/v3/df_monetary_{name}.parquet"
    df_monetary.to_parquet(output_path)

    # clean up memory
    del df_filtered
    del df_monetary
    gc.collect()

    print(f"Finished {name}, saved to {output_path}\n")

Processing ever_alert_part_1...


Feature Config Progress: 100%|█████████████████████████████████████████████████████████| 18/18 [05:42<00:00, 19.04s/it]


Finished ever_alert_part_1, saved to data/feature_engineering/credit/v3/df_monetary_ever_alert_part_1.parquet

Processing ever_alert_part_2...


Feature Config Progress: 100%|█████████████████████████████████████████████████████████| 18/18 [05:10<00:00, 17.23s/it]


Finished ever_alert_part_2, saved to data/feature_engineering/credit/v3/df_monetary_ever_alert_part_2.parquet

Processing never_alert_part_1...


Feature Config Progress: 100%|█████████████████████████████████████████████████████████| 18/18 [07:44<00:00, 25.83s/it]


Finished never_alert_part_1, saved to data/feature_engineering/credit/v3/df_monetary_never_alert_part_1.parquet

Processing never_alert_part_2...


Feature Config Progress: 100%|█████████████████████████████████████████████████████████| 18/18 [02:18<00:00,  7.70s/it]


Finished never_alert_part_2, saved to data/feature_engineering/credit/v3/df_monetary_never_alert_part_2.parquet



In [ ]:
# monetary dataset
df_monetary_1 = pd.read_parquet(
    "data/feature_engineering/credit/v3/df_monetary_ever_alert_part_1.parquet"
)
df_monetary_2 = pd.read_parquet(
    "data/feature_engineering/credit/v3/df_monetary_ever_alert_part_2.parquet"
)
df_monetary_3 = pd.read_parquet(
    "data/feature_engineering/credit/v3/df_monetary_never_alert_part_1.parquet"
)
df_monetary_4 = pd.read_parquet(
    "data/feature_engineering/credit/v3/df_monetary_never_alert_part_2.parquet"
)
df_monetary = pd.concat(
    [df_monetary_1, df_monetary_2, df_monetary_3, df_monetary_4], axis=0
)
df_monetary.reset_index(drop=True, inplace=True)

del df_monetary_1
del df_monetary_2
del df_monetary_3
del df_monetary_4

df_monetary.to_parquet("data/feature_engineering/credit/v3/df_monetary.parquet")
del df_monetary

gc.collect()

## Unique Count

In [15]:
df_credit_filter["MCC Num"], uniques = df_credit_filter["MCC"].factorize()
df_credit_filter["PANNumber Num"], uniques = df_credit_filter["PANNumber"].factorize()

In [16]:
# loop over splits, filter dataframe, run feature generation, and save parquet
for (
    name,
    pan_list,
) in splits.items():
    print(f"Processing {name}...")
    df_filtered = df_credit_filter[df_credit_filter.PANNumber.isin(pan_list)]

    # run feature engineering
    df_unique_count = generate_rolling_features(
        df_filtered,
        datetime_col="Transaction Datetime",
        key_col="Transaction Serial No",
        features_config=unique_count_config,
    )

    # save parquet
    output_path = f"data/feature_engineering/credit/v3/df_unique_count_{name}.parquet"
    df_unique_count.to_parquet(output_path)

    # clean up memory
    del df_filtered
    del df_unique_count
    gc.collect()

    print(f"Finished {name}, saved to {output_path}\n")

Processing ever_alert_part_1...


Feature Config Progress: 100%|███████████████████████████████████████████████████████| 5/5 [1:55:24<00:00, 1384.80s/it]


Finished ever_alert_part_1, saved to data/feature_engineering/credit/v3/df_unique_count_ever_alert_part_1.parquet

Processing ever_alert_part_2...


Feature Config Progress: 100%|███████████████████████████████████████████████████████| 5/5 [1:42:27<00:00, 1229.59s/it]


Finished ever_alert_part_2, saved to data/feature_engineering/credit/v3/df_unique_count_ever_alert_part_2.parquet

Processing never_alert_part_1...


Feature Config Progress: 100%|███████████████████████████████████████████████████████| 5/5 [3:01:36<00:00, 2179.22s/it]


Finished never_alert_part_1, saved to data/feature_engineering/credit/v3/df_unique_count_never_alert_part_1.parquet

Processing never_alert_part_2...


Feature Config Progress: 100%|██████████████████████████████████████████████████████████| 5/5 [26:26<00:00, 317.39s/it]


Finished never_alert_part_2, saved to data/feature_engineering/credit/v3/df_unique_count_never_alert_part_2.parquet



In [17]:
# unique count dataset
df_unique_cnt_1 = pd.read_parquet(
    "data/feature_engineering/credit/v3/df_unique_count_ever_alert_part_1.parquet"
)
df_unique_cnt_2 = pd.read_parquet(
    "data/feature_engineering/credit/v3/df_unique_count_ever_alert_part_2.parquet"
)
df_unique_cnt_3 = pd.read_parquet(
    "data/feature_engineering/credit/v3/df_unique_count_never_alert_part_1.parquet"
)
df_unique_cnt_4 = pd.read_parquet(
    "data/feature_engineering/credit/v3/df_unique_count_never_alert_part_2.parquet"
)
df_unique_cnt = pd.concat(
    [df_unique_cnt_1, df_unique_cnt_2, df_unique_cnt_3, df_unique_cnt_4], axis=0
)
df_unique_cnt.reset_index(drop=True, inplace=True)

del df_unique_cnt_1
del df_unique_cnt_2
del df_unique_cnt_3
del df_unique_cnt_4

df_unique_cnt.to_parquet("data/feature_engineering/credit/v3/df_unique_cnt.parquet")
del df_unique_cnt

gc.collect()

0